# IITM Deep Generative Models — Vanilla GAN: Mathematics ↔ Code

## How to use this notebook

This notebook keeps the **original Vanilla GAN implementation** and adds professor-style explanations around it.

The goal is not merely to run the code. The goal is to understand the mathematical story from the lecture PDF and see exactly where each mathematical object appears in Python/PyTorch.

The central pipeline is:

\[
Z\sim P_Z=N(0,I)
\quad\longrightarrow\quad
X_{\text{fake}}=G_\theta(Z)
\]

The generator induces a model distribution \(P_\theta\). The real MNIST data come from the unknown data distribution \(P_X\).

The GAN introduces a discriminator \(D_w\) that tries to distinguish

\[
X\sim P_X
\]

from

\[
G_\theta(Z)\sim P_\theta.
\]

The lecture's mathematical viewpoint is:

\[
\boxed{
\text{learn }P_\theta\text{ so that }P_\theta\approx P_X
}
\]

and GAN training realizes this through an adversarial/variational objective involving a discriminator.

### The one sentence to remember

> **The generator learns a mapping from simple noise to samples that look like real data; the discriminator provides the learning signal telling the generator how distinguishable its samples are from real data.**

# 1. First connect the lecture mathematics to this notebook

The lecture starts from data

\[
D=[X_1,\ldots,X_n],\qquad X_i\overset{iid}{\sim}P_X,
\]

where \(P_X\) is unknown.

For this notebook:

- \(X_i\) is an MNIST digit image.
- Each image has shape \(28\times 28\).
- After flattening, the data vector has dimension

\[
d=28\times 28=784.
\]

The notebook chooses a simple latent variable

\[
Z\sim N(0,I_{100}),
\]

so the latent dimension is 100.

The generator is therefore a neural network

\[
\boxed{
G_\theta:\mathbb R^{100}\to\mathbb R^{784}
}
\]

and the generated sample is

\[
\boxed{
X_{\text{fake}}=G_\theta(Z).
}
\]

Because the output layer uses `Tanh`, the generated coordinates lie approximately in \([-1,1]\), which matches the normalized MNIST data.

The discriminator is

\[
\boxed{
D_w:\mathbb R^{784}\to(0,1),
}
\]

where \(D_w(x)\) is interpreted as the probability that \(x\) is a **real** sample rather than a generated sample.

---

## The training game

For the discriminator, the mathematical objective represented by the code is the binary cross-entropy form

\[
\mathcal L_D(w;\theta)
=
-\mathbb E_{X\sim P_X}\log D_w(X)
-\mathbb E_{Z\sim P_Z}\log\left(1-D_w(G_\theta(Z))\right).
\]

The discriminator **minimizes** this loss.

Equivalently, it **maximizes**

\[
\mathbb E_{P_X}\log D_w(X)
+
\mathbb E_{P_Z}\log(1-D_w(G_\theta(Z))).
\]

This is the adversarial objective shown in the GAN part of the lecture.

The generator code uses the practical non-saturating objective

\[
\boxed{
\mathcal L_G(\theta;w)
=
-\mathbb E_{Z\sim P_Z}
\log D_w(G_\theta(Z)).
}
\]

The generator therefore tries to make the discriminator output \(1\) on generated samples.

**Important:** this generator loss is a practical implementation choice. It is not the literal \(\log(1-D(G(z)))\) minimax generator loss; it has the same adversarial goal but gives a stronger gradient when the discriminator is initially very confident.

# 2. Original Cell 0 — imports and computation device

Below is the original code, unchanged.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 0 — line-by-line mathematical explanation

### `import torch`

PyTorch provides tensors and automatic differentiation.

Mathematically, almost every object in the GAN is represented as a tensor:

- a real image \(x\in\mathbb R^{784}\),
- a latent vector \(z\in\mathbb R^{100}\),
- generator parameters \(\theta\),
- discriminator parameters \(w\),
- gradients such as \(\nabla_\theta \mathcal L_G\).

The most important idea is that PyTorch can compute derivatives automatically.

---

### `import torch.nn as nn`

This imports the neural-network building blocks.

For example:

\[
\texttt{nn.Linear(in,out)}
\]

represents an affine map

\[
h=W x+b.
\]

So the neural network in the mathematics is implemented by composing these layers.

---

### `import torch.optim as optim`

This gives optimization algorithms such as Adam.

The lecture asks:

\[
\theta^*=\arg\min_\theta \text{divergence}.
\]

In practice, we usually cannot solve that optimization in closed form. Instead we perform iterative gradient-based updates such as

\[
\theta_{k+1}
=
\theta_k-\eta\,\widehat{\nabla_\theta \mathcal L_G}.
\]

Adam is the optimizer used here.

---

### `import torchvision`

Provides standard computer-vision datasets and utilities.

Here it supplies MNIST and image-grid utilities.

---

### `import torchvision.transforms as transforms`

Provides transformations applied to raw MNIST images.

---

### `from torch.utils.data import DataLoader`

`DataLoader` turns the dataset into mini-batches.

Instead of using all \(n\) training examples at once, we repeatedly work with batches

\[
\{x_1,\ldots,x_B\}.
\]

This is the empirical approximation to expectations such as

\[
\mathbb E_{X\sim P_X}[f(X)]
\approx
\frac1B\sum_{i=1}^B f(x_i).
\]

---

### `import matplotlib.pyplot as plt`

Only for visualizing generated images. It is not part of the GAN mathematics.

---

### `device = torch.device(...)`

This chooses CPU or GPU.

It does **not** change the mathematical model. It changes where the tensor calculations are performed.

---

### `print(...)`

This only reports the chosen device.

### Key lesson from Cell 0

The lecture gives us an abstract optimization problem. PyTorch gives us the numerical machinery:

\[
\boxed{
\text{mathematical objective}
\;\longrightarrow\;
\text{tensor computation}
\;\longrightarrow\;
\text{automatic differentiation}
\;\longrightarrow\;
\text{optimizer update}.
}
\]

# 3. Original Cell 1 — data preprocessing and mini-batches

In [ ]:
# Transform: normalize images between [-1, 1] (because Tanh will be used at output)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load MNIST
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

## Cell 1 — line-by-line mathematical explanation

### `transform = transforms.Compose([...])`

This combines several preprocessing operations into one transformation.

---

### `transforms.ToTensor()`

MNIST pixels are stored as integer intensities.

`ToTensor()` converts the image into a floating-point tensor and scales pixel values to approximately

\[
[0,1].
\]

So a pixel \(p\in[0,255]\) becomes roughly

\[
x=\frac{p}{255}.
\]

---

### `transforms.Normalize((0.5,), (0.5,))`

The transformation is

\[
x_{\text{norm}}
=
\frac{x-0.5}{0.5}
=
2x-1.
\]

Therefore

\[
x\in[0,1]
\quad\Rightarrow\quad
x_{\text{norm}}\in[-1,1].
\]

This is why the generator later uses `Tanh`.

Since

\[
\tanh(u)\in(-1,1),
\]

the output range of the generator is compatible with the range of the real images.

This is an important implementation detail: **the real-data distribution and generator-output distribution should live in the same numerical range.**

---

### `train_dataset = torchvision.datasets.MNIST(...)`

This loads the samples

\[
x_1,\ldots,x_n
\]

from MNIST.

The notebook therefore instantiates the abstract lecture object

\[
D=[X_1,\ldots,X_n],\qquad X_i\sim P_X.
\]

Here \(P_X\) is the unknown distribution of MNIST images that we want the generator to imitate.

---

### `DataLoader(..., batch_size=128, shuffle=True)`

A batch contains up to

\[
B=128
\]

samples.

The GAN does not explicitly compute an integral such as

\[
\mathbb E_{P_X}[f(X)].
\]

Instead, it approximates it by a mini-batch average:

\[
\boxed{
\mathbb E_{P_X}[f(X)]
\approx
\frac1B\sum_{i=1}^B f(x_i).
}
\]

`shuffle=True` changes the order of examples across epochs, which is useful for stochastic optimization.

### Important connection

The PDF talks about distributions and expectations.

The code works with finite batches.

The bridge is:

\[
\boxed{
\text{expectation under a distribution}
\;\approx\;
\text{empirical average over sampled data}.
}
\]

# 4. Original Cell 2 — Generator network

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, img_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, img_dim),
            nn.Tanh()  # Because we normalized images to [-1, 1]
        )

    def forward(self, z):
        return self.model(z)

## Cell 2 — line-by-line mathematical explanation

### `class Generator(nn.Module):`

This defines the function

\[
G_\theta(z).
\]

The subscript \(\theta\) means that the function is controlled by trainable neural-network parameters.

---

### `def __init__(self, noise_dim, img_dim):`

The network needs to know:

- `noise_dim = 100`: dimension of \(z\),
- `img_dim = 784`: dimension of the flattened generated image.

Mathematically,

\[
z\in\mathbb R^{100},
\qquad
G_\theta(z)\in\mathbb R^{784}.
\]

---

### `super(...)`

Standard PyTorch initialization for a neural-network module.

---

### `self.model = nn.Sequential(...)`

This says:

> Apply the following transformations one after another.

Mathematically, the generator is a composition of functions:

\[
G_\theta
=
\phi_5\circ\phi_4\circ\phi_3\circ\phi_2\circ\phi_1.
\]

---

### `nn.Linear(noise_dim, 256)`

First affine transformation:

\[
h_1=W_1z+b_1.
\]

Dimensions:

\[
W_1\in\mathbb R^{256\times100}.
\]

---

### `nn.ReLU(True)`

Applies

\[
\operatorname{ReLU}(u)=\max(0,u)
\]

coordinate-wise:

\[
h_1'=\operatorname{ReLU}(h_1).
\]

This introduces the nonlinearity.

Without nonlinearities, a stack of linear layers would still collapse to one linear transformation.

---

### `nn.Linear(256, 512)`

\[
h_2=W_2h_1'+b_2.
\]

---

### `nn.ReLU(True)`

\[
h_2'=\operatorname{ReLU}(h_2).
\]

---

### `nn.Linear(512, 1024)`

\[
h_3=W_3h_2'+b_3.
\]

---

### `nn.ReLU(True)`

\[
h_3'=\operatorname{ReLU}(h_3).
\]

---

### `nn.Linear(1024, img_dim)`

The network maps to

\[
\mathbb R^{784}.
\]

Thus the generated object is a flattened \(28\times28\) image.

---

### `nn.Tanh()`

The final output is

\[
x_{\text{fake}}
=
\tanh(h_4).
\]

Therefore each output coordinate lies approximately in

\[
(-1,1).
\]

This matches the normalization applied to the real data.

---

### `def forward(self, z):`

This defines the mathematical function evaluated when we write

```python
generator(z)
```

which means

\[
\boxed{
x_{\text{fake}}=G_\theta(z).
}
\]

---

### `return self.model(z)`

Actually performs the composition of all the layers.

## What the generator means mathematically

This entire cell implements

\[
\boxed{
G_\theta:\mathbb R^{100}\to[-1,1]^{784}.
}
\]

If

\[
Z\sim N(0,I_{100}),
\]

then

\[
X_{\text{fake}}=G_\theta(Z)
\]

is a random variable whose distribution depends on \(\theta\).

That induced distribution is what the lecture calls the model distribution \(P_\theta\).

### The most important conceptual point

The generator does **not** directly store a probability table for images.

It stores a function \(G_\theta\).

The probability distribution arises because we feed random \(Z\) through that function:

\[
Z\sim P_Z
\quad\overset{G_\theta}{\longrightarrow}\quad
X_{\text{fake}}\sim P_\theta.
\]

That is exactly the connection between the PDF's latent-variable picture and the code.

# 5. Original Cell 3 — Discriminator network

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()  # Outputs probability between 0 and 1
        )

    def forward(self, img):
        return self.model(img)

## Cell 3 — line-by-line mathematical explanation

### `class Discriminator(nn.Module):`

This defines the discriminator

\[
D_w(x),
\]

where \(w\) denotes its trainable parameters.

---

### `nn.Linear(img_dim, 512)`

Input:

\[
x\in\mathbb R^{784}.
\]

First hidden representation:

\[
h_1=W_1x+b_1.
\]

---

### `nn.LeakyReLU(0.2, inplace=True)`

This is a variant of ReLU:

\[
\operatorname{LReLU}(u)
=
\begin{cases}
u,&u>0,\\
0.2u,&u\le0.
\end{cases}
\]

It keeps a nonzero slope on the negative side.

---

### `nn.Linear(512, 256)`

\[
h_2=W_2h_1+b_2.
\]

---

### `nn.LeakyReLU(...)`

Again apply the nonlinear activation.

---

### `nn.Linear(256, 1)`

The network ends with one scalar logit:

\[
a(x)\in\mathbb R.
\]

---

### `nn.Sigmoid()`

Transforms the scalar into

\[
D_w(x)=\sigma(a(x))
=
\frac{1}{1+e^{-a(x)}}.
\]

Hence

\[
0<D_w(x)<1.
\]

We interpret

\[
D_w(x)\approx P(\text{real}\mid x).
\]

This is why the lecture calls \(D_w\) a classifier/discriminator.

---

### `forward(self, img)`

This defines how the discriminator is evaluated on an image.

### `return self.model(img)`

So

\[
\boxed{
\text{image }x
\longrightarrow
D_w(x)\in(0,1).
}
\]

### Connection to the lecture

The PDF has the structure

\[
D_w:\mathcal X\to[0,1].
\]

The code implements exactly that idea with a neural network followed by a sigmoid.

# 6. Original Cell 4 — dimensions, models, optimizers, loss

In [ ]:
noise_dim = 100
img_dim = 28 * 28

# Models
generator = Generator(noise_dim, img_dim).to(device)
discriminator = Discriminator(img_dim).to(device)

# Optimizers
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002)
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002)

# Loss
criterion = nn.BCELoss()

## Cell 4 — line-by-line mathematical explanation

### `noise_dim = 100`

This chooses the latent dimension

\[
z\in\mathbb R^{100}.
\]

---

### `img_dim = 28 * 28`

MNIST images are \(28\times28\), hence

\[
d=784.
\]

---

### `generator = Generator(...).to(device)`

This creates the parameterized function

\[
G_\theta.
\]

The network parameters \(\theta\) are now actual tensors that can be optimized.

---

### `discriminator = Discriminator(...).to(device)`

Creates

\[
D_w.
\]

Its parameters \(w\) are separately optimized.

---

### `g_optimizer = optim.Adam(generator.parameters(), lr=0.0002)`

This says:

> Update only \(\theta\) using the generator loss.

Conceptually,

\[
\theta\leftarrow\theta-\eta\,\widehat{\nabla_\theta\mathcal L_G}.
\]

---

### `d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002)`

Similarly,

\[
w\leftarrow w-\eta\,\widehat{\nabla_w\mathcal L_D}.
\]

Notice something important:

The lecture may write the discriminator as a **maximization** problem, but the code minimizes binary cross-entropy.

These are equivalent up to a minus sign:

\[
\max_w J_D(w)
\quad\Longleftrightarrow\quad
\min_w[-J_D(w)].
\]

---

### `criterion = nn.BCELoss()`

Binary cross-entropy for one example is

\[
\operatorname{BCE}(y,\hat y)
=
-\left[
y\log\hat y
+
(1-y)\log(1-\hat y)
\right].
\]

For a real image, \(y=1\):

\[
\operatorname{BCE}(1,D_w(x))
=
-\log D_w(x).
\]

For a fake image, \(y=0\):

\[
\operatorname{BCE}(0,D_w(x))
=
-\log(1-D_w(x)).
\]

Therefore this single PyTorch loss function implements both pieces of the GAN discriminator objective.

---

# The key mathematical bridge

The lecture's discriminator objective is of the form

\[
J_D(\theta,w)
=
\mathbb E_{P_X}[\log D_w(X)]
+
\mathbb E_{P_Z}
[\log(1-D_w(G_\theta(Z)))].
\]

The code minimizes

\[
\mathcal L_D=-J_D.
\]

So:

\[
\boxed{
\texttt{BCELoss}
\quad\leftrightarrow\quad
\text{GAN discriminator log-likelihood}.
}
\]

# 7. Original Cell 5 — visualization of the generator

In [ ]:
def show_generated_images(epoch, generator, fixed_noise):
    generator.eval()
    with torch.no_grad():
        fake_imgs = generator(fixed_noise).reshape(-1, 1, 28, 28)
        fake_imgs = fake_imgs * 0.5 + 0.5  # De-normalize

    grid = torchvision.utils.make_grid(fake_imgs, nrow=8)
    plt.figure(figsize=(8,8))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.title(f'Generated Images at Epoch {epoch}')
    plt.axis('off')
    plt.show()
    generator.train()

## Cell 5 — line-by-line mathematical explanation

### `def show_generated_images(...)`

This function is **not training** the GAN. It is only for observing the current generator.

---

### `generator.eval()`

Switches the model to evaluation mode.

For this particular generator, there are no BatchNorm/Dropout layers, so this has little practical effect here, but it is conventional.

---

### `with torch.no_grad():`

Tells PyTorch not to construct a gradient computation graph.

We only want to sample:

\[
z\sim P_Z
\quad\rightarrow\quad
G_\theta(z).
\]

We are not updating \(\theta\).

---

### `fake_imgs = generator(fixed_noise)`

This computes

\[
X_{\text{fake}}=G_\theta(Z_{\text{fixed}}).
\]

The important word is **fixed**. Using the same noise vectors every time lets us compare the generator's progress over training.

---

### `.reshape(-1,1,28,28)`

The generator outputs vectors in \(\mathbb R^{784}\).

This reshapes each vector back into image form:

\[
784\longrightarrow 1\times28\times28.
\]

It changes representation, not the underlying values.

---

### `fake_imgs = fake_imgs * 0.5 + 0.5`

Recall normalization:

\[
x_{\text{norm}}=2x-1.
\]

Solving for the original scale:

\[
\boxed{
x=\frac{x_{\text{norm}}+1}{2}
=0.5x_{\text{norm}}+0.5.
}
\]

Therefore this line reverses the preprocessing before display.

---

### `make_grid(...)`

Arranges multiple generated samples into one visualization.

---

### `plt.imshow(...)`

Displays the images.

---

### `plt.title(...)`, `axis('off')`, `show()`

Only visualization.

---

### `generator.train()`

Returns the model to training mode.

## Mathematical meaning of the whole function

It performs:

\[
\boxed{
Z_1,\ldots,Z_{64}\sim P_Z
\quad\longrightarrow\quad
G_\theta(Z_1),\ldots,G_\theta(Z_{64})
}
\]

and lets us visually inspect the samples from the current learned model \(P_\theta\).

# 8. Original Cell 6 — the complete GAN training loop

In [ ]:
def train_gan(train_loader, num_epochs, mode="one_one"):
    fixed_noise = torch.randn(64, noise_dim).to(device)  # For consistent visualization

    for epoch in range(num_epochs):
        for batch_idx, (real, _) in enumerate(train_loader):
            batch_size = real.size(0)
            real = real.view(batch_size, -1).to(device)

            ### Create real and fake labels
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            ### ====================================
            ### 1. Train Discriminator
            ### ====================================

            # Real images
            outputs = discriminator(real)
            d_loss_real = criterion(outputs, real_labels)
            real_score = outputs

            # Fake images
            z = torch.randn(batch_size, noise_dim).to(device)
            fake = generator(z)
            outputs = discriminator(fake.detach())  # detach so gradients don't flow into generator
            d_loss_fake = criterion(outputs, fake_labels)
            fake_score = outputs

            # Total discriminator loss
            d_loss = d_loss_real + d_loss_fake

            discriminator.zero_grad()
            d_loss.backward()
            d_optimizer.step()

            ### ====================================
            ### 2. Train Generator
            ### ====================================
            # Generate fake images again
            z = torch.randn(batch_size, noise_dim).to(device)
            fake = generator(z)
            outputs = discriminator(fake)

            g_loss = criterion(outputs, real_labels)  # trick discriminator

            generator.zero_grad()
            g_loss.backward()
            g_optimizer.step()

            ### ====================================
            ### 3. Modify according to mode
            ### ====================================
            if mode == "five_gen_one_disc":
                for _ in range(4):  # already did 1 step above
                    z = torch.randn(batch_size, noise_dim).to(device)
                    fake = generator(z)
                    outputs = discriminator(fake)
                    g_loss = criterion(outputs, real_labels)

                    generator.zero_grad()
                    g_loss.backward()
                    g_optimizer.step()

            if mode == "five_disc_one_gen":
                for _ in range(4):  # already did 1 step above
                    # Real
                    outputs = discriminator(real)
                    d_loss_real = criterion(outputs, real_labels)

                    # Fake
                    z = torch.randn(batch_size, noise_dim).to(device)
                    fake = generator(z)
                    outputs = discriminator(fake.detach())
                    d_loss_fake = criterion(outputs, fake_labels)

                    d_loss = d_loss_real + d_loss_fake

                    discriminator.zero_grad()
                    d_loss.backward()
                    d_optimizer.step()

        if (epoch+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], D_loss: {d_loss.item():.4f}, G_loss: {g_loss.item():.4f}")
            show_generated_images(epoch+1, generator, fixed_noise)

# Cell 6 — line-by-line mathematical explanation

This is the most important cell in the entire notebook.

It implements the alternating optimization:

\[
\boxed{
\text{update discriminator}
\;\longrightarrow\;
\text{update generator}
\;\longrightarrow\;
\text{repeat}.
}
\]

---

## Function definition

### `def train_gan(train_loader, num_epochs, mode="one_one"):`

This creates a training routine.

- `train_loader` provides samples from \(P_X\),
- `num_epochs` says how many passes to make through the dataset,
- `mode` controls how many discriminator/generator updates are performed.

---

## Fixed noise

### `fixed_noise = torch.randn(64, noise_dim).to(device)`

This samples

\[
Z_{\text{fixed}}
\sim N(0,I).
\]

It is only for visualization.

It is **not** the random noise used for the gradient updates.

---

## Epoch loop

### `for epoch in range(num_epochs):`

Repeat optimization over the training data.

---

## Mini-batch loop

### `for batch_idx, (real, _) in enumerate(train_loader):`

The batch contains real images \(x_1,\ldots,x_B\).

The `_` is the MNIST digit label. This Vanilla GAN does not use class labels, so the label is ignored.

This means the notebook is learning the unconditional distribution

\[
P_X
\]

rather than a conditional distribution \(P_{X\mid Y}\).

---

## Batch size

### `batch_size = real.size(0)`

Let the current batch contain \(B\) samples.

---

## Flattening

### `real = real.view(batch_size, -1).to(device)`

Each image changes from

\[
1\times28\times28
\]

to

\[
784.
\]

So each real sample becomes

\[
x_i\in\mathbb R^{784}.
\]

---

# 8.1 Construct the labels

### `real_labels = torch.ones(batch_size, 1)`

For every real example,

\[
y=1.
\]

### `fake_labels = torch.zeros(batch_size, 1)`

For every generated example,

\[
y=0.
\]

So the discriminator is solving an ordinary binary classification problem:

\[
\text{real}\to1,
\qquad
\text{fake}\to0.
\]

---

# 8.2 Train the discriminator on real images

### `outputs = discriminator(real)`

For real samples:

\[
D_w(x_i).
\]

---

### `d_loss_real = criterion(outputs, real_labels)`

Because the target is \(1\),

\[
\operatorname{BCE}(1,D_w(x_i))
=
-\log D_w(x_i).
\]

Over a batch,

\[
\boxed{
L_D^{\text{real}}
\approx
-\frac1B\sum_{i=1}^B
\log D_w(x_i).
}
\]

The discriminator wants \(D_w(x_i)\) close to 1.

---

### `real_score = outputs`

Stores the discriminator's scores for possible monitoring.

No mathematical update occurs here.

---

# 8.3 Train the discriminator on fake images

### `z = torch.randn(batch_size, noise_dim).to(device)`

Sample fresh latent variables

\[
z_1,\ldots,z_B
\overset{iid}{\sim}
N(0,I).
\]

---

### `fake = generator(z)`

Generate

\[
\boxed{
\tilde x_i=G_\theta(z_i).
}
\]

These samples approximately follow the generator distribution \(P_\theta\).

---

### `outputs = discriminator(fake.detach())`

Compute

\[
D_w(G_\theta(z_i)).
\]

But `detach()` is crucial.

It means:

> When computing the discriminator update, do not send gradients through the generator.

Mathematically, during this step we treat \(\theta\) as fixed and optimize only \(w\).

This matches the lecture's alternating optimization:

\[
\boxed{
\theta\text{ fixed},\qquad
\min_w\mathcal L_D(\theta,w).
}
\]

---

### `d_loss_fake = criterion(outputs, fake_labels)`

Because the target is \(0\),

\[
\operatorname{BCE}(0,D_w(G_\theta(z)))
=
-\log(1-D_w(G_\theta(z))).
\]

Thus

\[
\boxed{
L_D^{\text{fake}}
\approx
-\frac1B\sum_{i=1}^B
\log\left(1-D_w(G_\theta(z_i))\right).
}
\]

The discriminator wants to say:

\[
D_w(G_\theta(z_i))\approx0.
\]

---

### `fake_score = outputs`

Stores the discriminator scores on fake samples.

---

# 8.4 Total discriminator loss

### `d_loss = d_loss_real + d_loss_fake`

Therefore

\[
\boxed{
L_D
=
-\frac1B\sum_i\log D_w(x_i)
-\frac1B\sum_i\log\left(1-D_w(G_\theta(z_i))\right).
}
\]

This is the negative empirical version of the GAN discriminator objective from the lecture.

Equivalently,

\[
\max_w
\left[
\frac1B\sum_i\log D_w(x_i)
+
\frac1B\sum_i\log(1-D_w(G_\theta(z_i)))
\right].
\]

---

# 8.5 Backpropagation and discriminator update

### `discriminator.zero_grad()`

Clear old gradients.

Otherwise PyTorch would accumulate gradients from previous steps.

---

### `d_loss.backward()`

Compute

\[
\nabla_w L_D.
\]

Because the generated tensor was detached, the gradient does not propagate into \(\theta\).

---

### `d_optimizer.step()`

Update discriminator parameters approximately as

\[
\boxed{
w
\leftarrow
w-\eta_D\nabla_wL_D.
}
\]

Adam actually uses an adaptive version of this update.

---

# 8.6 Train the generator

Now we reverse the roles.

### `z = torch.randn(batch_size, noise_dim).to(device)`

Draw new noise.

\[
z_i\sim N(0,I).
\]

---

### `fake = generator(z)`

Generate

\[
\tilde x_i=G_\theta(z_i).
\]

---

### `outputs = discriminator(fake)`

Compute

\[
D_w(G_\theta(z_i)).
\]

This time **there is no `detach()`**.

That is intentional.

We want

\[
\boxed{
\frac{\partial D_w(G_\theta(z))}
{\partial\theta}
}
\]

to be available.

The gradient must pass through the discriminator into the generator.

---

### `g_loss = criterion(outputs, real_labels)`

This is subtle and extremely important.

The generated image is given target label \(1\).

So the generator is asking:

> "Make the discriminator believe my generated sample is real."

The resulting loss is

\[
\boxed{
L_G
=
-\frac1B\sum_i
\log D_w(G_\theta(z_i)).
}
\]

This is the **non-saturating generator loss**.

---

### `generator.zero_grad()`

Clear previous generator gradients.

---

### `g_loss.backward()`

Compute

\[
\nabla_\theta L_G.
\]

The chain rule is conceptually

\[
\frac{\partial L_G}{\partial\theta}
=
\frac{\partial L_G}{\partial D}
\frac{\partial D}{\partial G_\theta}
\frac{\partial G_\theta}{\partial\theta}.
\]

This one line is the mathematical heart of how the discriminator teaches the generator.

---

### `g_optimizer.step()`

Update

\[
\boxed{
\theta
\leftarrow
\theta-\eta_G\nabla_\theta L_G.
}
\]

The generator changes so that future generated samples are more likely to receive high discriminator scores.

---

# 8.7 Changing the number of updates

The notebook then experiments with three training schedules.

## `mode == "five_gen_one_disc"`

The discriminator has already received one update.

Then:

```python
for _ in range(4):
```

adds four more generator updates.

So in this batch:

\[
\boxed{
1\text{ discriminator update}
\quad\text{vs.}\quad
5\text{ generator updates}.
}
\]

During those extra generator updates, the discriminator parameters are held fixed.

---

## `mode == "five_disc_one_gen"`

The generator has already received one update.

Then four additional discriminator updates are performed.

So:

\[
\boxed{
5\text{ discriminator updates}
\quad\text{vs.}\quad
1\text{ generator update}.
}
\]

---

## Why would we change the ratio?

GAN training is an adversarial optimization problem.

The two networks have competing objectives.

If one becomes dramatically stronger than the other, optimization may become difficult.

Therefore the update schedule itself is an experimental hyperparameter.

---

# 8.8 Epoch-end reporting

### `if (epoch+1) % 10 == 0:`

Every 10 epochs, report the current losses and show generated images.

---

### `print(...)`

The displayed losses are useful diagnostics.

Be careful: these are the losses from the **last processed mini-batch of that epoch**, not the average over the whole epoch.

---

### `show_generated_images(...)`

Samples from the current generator and visualizes them.

This is where the abstract \(P_\theta\) becomes something you can actually see.

---

# The entire Cell 6 in one mathematical block

For each mini-batch:

### Step A — discriminator

\[
\boxed{
w
\leftarrow
\operatorname{Adam}
\left(
w,
\nabla_w
\left[
-\mathbb E_{P_X}\log D_w(X)
-\mathbb E_{P_Z}\log(1-D_w(G_\theta(Z)))
\right]
\right).
}
\]

### Step B — generator

\[
\boxed{
\theta
\leftarrow
\operatorname{Adam}
\left(
\theta,
\nabla_\theta
\left[
-\mathbb E_{P_Z}\log D_w(G_\theta(Z))
\right]
\right).
}
\]

Thus the code is implementing:

\[
\boxed{
P_Z
\xrightarrow{G_\theta}
P_\theta
\quad\text{and}\quad
D_w\text{ tries to distinguish }P_X\text{ from }P_\theta.
}
\]

That is the practical implementation of the lecture's GAN idea.

# 9. Original Cell 7 — three training experiments

In [ ]:
# 1. Standard: One step Generator, One step Discriminator
print("Training: 1-step Gen, 1-step Disc")
train_gan(train_loader, num_epochs=50, mode="one_one")

# 2. 5 Generator steps, 1 Discriminator step
print("Training: 5-step Gen, 1-step Disc")
train_gan(train_loader, num_epochs=50, mode="five_gen_one_disc")

# 3. 5 Discriminator steps, 1 Generator step
print("Training: 5-step Disc, 1-step Gen")
train_gan(train_loader, num_epochs=50, mode="five_disc_one_gen")

## Cell 7 — what each experiment means mathematically

### Experiment 1

```text
1-step Generator, 1-step Discriminator
```

This approximates the basic alternating game:

\[
\boxed{
w\leftarrow w-\eta_D\nabla_wL_D,
\qquad
\theta\leftarrow\theta-\eta_G\nabla_\theta L_G.
}
\]

---

### Experiment 2

```text
5-step Generator, 1-step Discriminator
```

Now the generator gets five optimization updates for every discriminator update.

Conceptually:

\[
\theta
\text{ moves more aggressively before }w\text{ moves again}.
\]

---

### Experiment 3

```text
5-step Discriminator, 1-step Generator
```

Now the discriminator receives five updates for every generator update.

Conceptually:

\[
w
\text{ is trained much more strongly before }\theta\text{ moves again}.
\]

---

# IMPORTANT implementation caveat

The three experiments in the notebook are **not independent experiments**.

The same objects are reused:

```python
generator
discriminator
g_optimizer
d_optimizer
```

The first 50-epoch run trains them.

Then the second 50-epoch run starts from those already-trained parameters.

Then the third 50-epoch run starts from the second run.

Therefore the comparison is actually closer to:

\[
\text{50 epochs}
\rightarrow
\text{another 50 epochs}
\rightarrow
\text{another 50 epochs},
\]

with different update schedules.

It is **not**:

\[
\text{fresh model A}
\quad\text{vs.}\quad
\text{fresh model B}
\quad\text{vs.}\quad
\text{fresh model C}.
\]

Also, Adam keeps optimizer state, so even the optimizer's internal momentum/second-moment estimates are carried between experiments.

For a scientifically clean comparison, each schedule should start from a newly initialized generator, discriminator, and optimizer state.

This is a critique of the implementation, not a change to the original code.

# 10. The complete PDF ↔ code dictionary

| Mathematical object | In the lecture | In this notebook |
|---|---|---|
| Real data distribution | \(P_X\) | MNIST training data |
| Real sample | \(X\sim P_X\) | `real` |
| Latent distribution | \(P_Z=N(0,I)\) | `torch.randn(...)` |
| Latent variable | \(Z\) | `z` |
| Generator | \(g_\theta(z)\) / \(G_\theta(z)\) | `generator(z)` |
| Generator parameters | \(\theta\) | `generator.parameters()` |
| Model distribution | \(P_\theta\) | distribution induced by `generator(z)` |
| Discriminator | \(D_w(x)\) | `discriminator(x)` |
| Discriminator parameters | \(w\) | `discriminator.parameters()` |
| Real target | \(1\) | `real_labels` |
| Fake target | \(0\) | `fake_labels` |
| Divergence/objective estimation | variational/discriminator objective | BCE GAN loss |
| Expectation | \(\mathbb E[\cdot]\) | mini-batch average inside `BCELoss` |
| Optimization | minimize/maximize objective | Adam `.step()` |
| Sampling | \(z\sim P_Z\to G_\theta(z)\) | `torch.randn` then `generator(z)` |

---

# 11. The entire GAN as one flow

You should now be able to draw this on a board:

\[
\boxed{
z\sim N(0,I)
}
\]

↓

\[
\boxed{
x_{\text{fake}}=G_\theta(z)
}
\]

↓

\[
\boxed{
x_{\text{fake}}\sim P_\theta
}
\]

while real data provide

\[
\boxed{
x_{\text{real}}\sim P_X.
}
\]

The discriminator receives both:

\[
D_w(x_{\text{real}})
\qquad\text{and}\qquad
D_w(x_{\text{fake}}).
\]

It is trained to satisfy

\[
D_w(x_{\text{real}})\to1,
\qquad
D_w(x_{\text{fake}})\to0.
\]

The generator sees the feedback through the discriminator and is trained so that

\[
D_w(G_\theta(z))\to1.
\]

Therefore the generator is continually changed so that its distribution \(P_\theta\) becomes harder to distinguish from \(P_X\).

---

# 12. Why `detach()` is the key line in the discriminator step

This deserves special attention.

Without

```python
fake.detach()
```

the discriminator backward pass would also send gradients into the generator.

Then the "discriminator update" would accidentally modify the generator computation graph as well.

With `detach()`:

\[
\boxed{
\text{during discriminator update: }\theta\text{ is treated as fixed}.
}
\]

During the generator update, we remove `detach()`:

\[
\boxed{
\text{during generator update: }w\text{ is treated as fixed, but gradients pass through }D_w\text{ into }\theta.
}
\]

This is exactly what alternating optimization requires.

---

# 13. Why the generator uses labels = 1

A beginner often finds this line confusing:

```python
g_loss = criterion(outputs, real_labels)
```

The generated images are fake, so why use the target `1`?

Because this is **not asking the discriminator to train**.

It is asking the generator:

> "Change your parameters so that the discriminator classifies your samples as real."

Mathematically:

\[
\min_\theta
-\mathbb E_Z
\log D_w(G_\theta(Z)).
\]

If \(D_w(G_\theta(z))\) increases toward \(1\), then

\[
-\log D_w(G_\theta(z))
\]

decreases toward \(0\).

Thus the generator improves its objective by fooling the discriminator.

# 14. A worked single-sample view

Forget the whole batch for a moment.

Suppose:

\[
z\sim N(0,I),
\]

and the generator produces

\[
x_f=G_\theta(z).
\]

Suppose the discriminator gives

\[
D_w(x_f)=0.2.
\]

### Discriminator's fake loss

Because the sample is fake:

\[
L_D^{fake}
=
-\log(1-0.2)
=
-\log(0.8).
\]

This is relatively small.

So the discriminator is doing a reasonable job.

### Generator's loss

The generator uses the target 1:

\[
L_G
=
-\log(0.2).
\]

This is large.

Therefore the generator gets a strong incentive to change \(\theta\) so that

\[
D_w(G_\theta(z))
\]

becomes larger.

Suppose training moves it to \(0.8\).

Then

\[
L_G=-\log(0.8),
\]

which is much smaller.

That is the adversarial mechanism in one number.

# 15. What you should understand before moving to the next lecture section

You should be able to answer all of these without looking at the code:

### Probability level

1. What is \(P_X\)?
2. What is \(P_Z\)?
3. Why is \(Z\) easy to sample?
4. How does \(G_\theta(Z)\) produce a new distribution \(P_\theta\)?

### Neural-network level

5. What exactly is \(G_\theta\)?
6. What exactly is \(D_w\)?
7. Why does the generator end with `Tanh`?
8. Why does the discriminator end with `Sigmoid`?

### Optimization level

9. What objective does the discriminator minimize?
10. What objective does the generator minimize?
11. Why does `detach()` appear only in the discriminator's fake-sample path?
12. Why are generator and discriminator updated separately?

### Mathematical foundation

13. How does a mini-batch approximate an expectation?
14. How does binary cross-entropy produce the terms
\[
-\log D_w(x)
\quad\text{and}\quad
-\log(1-D_w(G_\theta(z)))?
\]
15. Why is GAN training an adversarial optimization problem?

If these questions are clear, then the code is no longer a collection of PyTorch commands. It is an implementation of the mathematical model.

# 16. Final mental model

The entire notebook can be compressed to the following picture:

\[
\boxed{
\begin{array}{c}
Z\sim N(0,I)
\\[2mm]
\downarrow
\\[2mm]
G_\theta
\\[2mm]
\downarrow
\\[2mm]
X_{\text{fake}}\sim P_\theta
\end{array}
}
\qquad
\text{versus}
\qquad
\boxed{
X_{\text{real}}\sim P_X
}
\]

The discriminator learns

\[
\boxed{
\text{real}\rightarrow1,
\qquad
\text{fake}\rightarrow0
}
\]

and the generator learns

\[
\boxed{
\text{make fake look real}.
}
\]

Repeated alternation gives

\[
\boxed{
P_\theta \longrightarrow P_X
\quad\text{(ideally)}.
}
\]

This is the direct mathematical-to-implementation bridge for the Vanilla GAN notebook.